# 06 JEV and LAYA as judges
JEV and LAYA never hear the audio; they only reason over the evidence we send.

- **Section 1, JEV on one detector:** why FP1 adds nothing, and whether richer evidence helps. 1.1–1.3 build the
  evidence JEV could be given, 1.4 inspects what FP1 actually sent and got back, 1.5 tries richer evidence.
- **Section 2, arbitration between detectors:** when several detectors disagree, can a judge pick the right one?
  Tried with LAYA first (local, free), then JEV on the request that works best.

## Section 1: JEV on one detector's evidence

In [2]:
import hashlib
import json
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from audio_df.data.manifests import load_manifest
from audio_df.helpers.audio import load_audio
from audio_df.helpers.eval_audio import TRIM_RECIPE
from audio_df.helpers.metrics import compute_eer
from audio_df.helpers.research.runs import collect
from audio_df.modules.jev import JEVState, decide

SR, SETS, N_RANDOM, N_DISAGREE = 16000, ["itw", "asvspoof2021_hidden", "asvspoof2021df_hidden"], 10, 10
DETECTORS = {"a1": "A1 LFCC-GMM", "a2": "A2 WavLM", "d2 L6": "D2 WavLM+LFCC one-way", "d3 L6": "D3 WavLM+LFCC two-way"}
runs = {k[0]: v["scores"] for k, v in collect("../runs", ["a1", "a2", "d2", "d3"]).items()
        if k[1] == "trimmed" and k[0] in DETECTORS}

def platt(dev):  # dev score -> P(bona fide)
    return LogisticRegression().fit(dev[["score"]], dev.label == "bonafide")

calib = {m: platt(runs[m]["dev"]) for m in DETECTORS}
thr = {m: compute_eer(runs[m]["dev"].score, runs[m]["dev"].label, higher_is_spoof=False)[1] for m in DETECTORS}

d:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1 Clips
Per test set: 10 random clips plus 10 where D2 and A2 disagree at their dev thresholds, so there are mistakes to fix.
Results are reported per group; the disagreement group is deliberately hard.

In [3]:
rng = np.random.default_rng(1337)
pool = []
for s in SETS:
    merged = None
    for m in DETECTORS:
        part = runs[m][s][["clip_id", "label", "score"]].rename(columns={"score": m})
        merged = part if merged is None else merged.merge(part, on=["clip_id", "label"])
    merged["set"] = s
    disagree = (merged["d2 L6"] <= thr["d2 L6"]) != (merged["a2"] <= thr["a2"])
    pick_d = merged[disagree].sample(N_DISAGREE, random_state=1337).assign(group="disagree")
    pick_r = merged[~merged.clip_id.isin(pick_d.clip_id)].sample(N_RANDOM, random_state=1337).assign(group="random")
    pool += [pick_r, pick_d]
pool = pd.concat(pool, ignore_index=True)
for m in DETECTORS:
    pool[f"p_{m}"] = calib[m].predict_proba(pool[[m]].rename(columns={m: "score"}))[:, 1]
pool.groupby(["set", "group"]).label.value_counts().unstack(fill_value=0)

label                           bonafide  spoof
set                   group                    
asvspoof2021_hidden   disagree         5      5
                      random           0     10
asvspoof2021df_hidden disagree         1      9
                      random           0     10
itw                   disagree        10      0
                      random           9      1

## 1.2 Readable audio facts
What an analyst could read off the waveform: duration, leading silence (same rule as trimming), effective bandwidth
(highest frequency within 50 dB of the spectral peak: ~8 kHz for studio audio, ~4 kHz for telephone or narrowband)
and share of speech energy above 4 kHz (spoofs had less in notebook 03).

In [4]:
paths = {s: load_manifest(f"../data/manifests/{s}.csv", "..").set_index("utt_id").path for s in SETS}

def describe(path):
    x = load_audio(path, SR)
    speech = librosa.effects.split(x, top_db=TRIM_RECIPE["top_db"], frame_length=TRIM_RECIPE["frame_length"],
                                   hop_length=TRIM_RECIPE["hop_length"])
    voiced = np.concatenate([x[s:e] for s, e in speech]) if len(speech) else x
    power = (np.abs(librosa.stft(voiced, n_fft=512)) ** 2).mean(axis=1)  # average spectrum of speech
    freqs = librosa.fft_frequencies(sr=SR, n_fft=512)
    return {"duration_s": round(len(x) / SR, 2),
            "leading_silence_s": round(speech[0, 0] / SR, 2) if len(speech) else 0.0,
            # highest frequency within 50 dB of the spectral peak: ~8 kHz studio, ~4 kHz telephone/narrowband
            "bandwidth_khz": round(float(freqs[np.nonzero(power >= power.max() * 1e-5)[0].max()]) / 1000, 2),
            "energy_above_4khz_pct": round(100 * power[freqs >= 4000].sum() / power.sum(), 2)}

pool = pool.join(pd.DataFrame([describe(paths[r.set][r.clip_id]) for r in pool.itertuples()]))
pool.groupby("label")[["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]].median()

,duration_s,leading_silence_s,bandwidth_khz,energy_above_4khz_pct
label,,,,
bonafide,2.85,0.13,7.69,0.35
spoof,1.74,0.00,7.72,0.43


## 1.3 Rules without JEV, on the same clips
- **D2 alone:** its dev threshold.
- **WavLM average:** mean P(bona fide) of A2, D2, D3 ≥ 0.5.
- **Abstain on disagreement:** D2's verdict, or "uncertain" when D2 and A2 disagree.

In [5]:
def summarise(pred, keep=None):  # pred: "bonafide" / "spoof" / "uncertain" per clip; keep: rows to score
    keep = np.ones(len(pool), bool) if keep is None else keep
    pred, label = np.asarray(pred)[keep], pool.label.to_numpy()[keep]
    decided = pred != "uncertain"
    correct = (pred == label) & decided
    return pd.Series({"coverage": decided.mean(), "accuracy on decided": correct.sum() / max(decided.sum(), 1),
                      "errors": int((decided & ~correct).sum()), "abstained": int((~decided).sum())})

verdict = lambda p: np.where(p >= 0.5, "bonafide", "spoof")
d2 = np.where(pool["d2 L6"] > thr["d2 L6"], "bonafide", "spoof")
a2 = np.where(pool["a2"] > thr["a2"], "bonafide", "spoof")
policies = {"D2 alone": d2,
            "WavLM average": verdict(pool[["p_a2", "p_d2 L6", "p_d3 L6"]].mean(axis=1)),
            "abstain on disagreement": np.where(d2 == a2, d2, "uncertain")}
pd.DataFrame({k: summarise(v) for k, v in policies.items()}).T.round(3)

,coverage,accuracy on decided,errors,abstained
D2 alone,1.000,0.617,23.0,0.0
WavLM average,1.000,0.650,21.0,0.0
abstain on disagreement,0.467,0.821,5.0,32.0


## 1.4 What FP1 sent and got
FP1 (`uv run fp1`) asks JEV about one D2 score per clip: 20 clips from each of 9 test sets. Per test set: D2's
accuracy, how often JEV decided, how often JEV gave the same answer as D2, and the errors JEV corrected or
introduced.

In [2]:
FP1 = Path("../runs/E-20261028-081544-540928-fp1")
paired = pd.concat({p.stem.removeprefix("paired_"): pd.read_csv(p) for p in sorted(FP1.glob("paired_*.csv"))})
pd.DataFrame({s: {"clips": len(g), "D2 accuracy": (g.base == g.label).mean(),
                  "JEV coverage": (g.jev != "uncertain").mean(), "JEV = D2": (g.jev == g.base).mean(),
                  "corrected": int(((g.base != g.label) & (g.jev == g.label)).sum()),
                  "introduced": int(((g.base == g.label) & (g.jev != g.label) & (g.jev != "uncertain")).sum())}
              for s, g in paired.groupby(level=0)}).T.round(2)

,clips,D2 accuracy,JEV coverage,JEV = D2,corrected,introduced
asvspoof2021_eval,20.0,0.95,1.0,1.0,0.0,0.0
asvspoof2021_hidden,20.0,0.90,1.0,1.0,0.0,0.0
eval,20.0,0.90,1.0,1.0,0.0,0.0
eval_clean10k,20.0,0.90,1.0,1.0,0.0,0.0
eval_mp3_32k,20.0,0.95,1.0,1.0,0.0,0.0
eval_noise_10db,20.0,0.90,1.0,1.0,0.0,0.0
eval_opus_16k,20.0,0.90,1.0,1.0,0.0,0.0
eval_resample_8k,20.0,0.80,1.0,1.0,0.0,0.0
itw,20.0,0.50,1.0,1.0,0.0,0.0


In [3]:
request = json.loads(open(FP1 / "jev/requests.jsonl").readline())
response = json.loads(open(FP1 / "jev/responses.jsonl").readline())
print(json.dumps({"state": request["state"], "questions": list(request["questions"])}, indent=1))
print(json.dumps(response["raw_jev_response"]["answers"]["decision"], indent=1))

{
 "state": {
  "context": {
   "base_score": 0.995396077632904,
   "dev_threshold": -0.9904876351356506,
   "base_verdict": "bonafide",
   "signed_margin": 1.9858837127685547,
   "score_direction": "higher means bonafide"
  },
  "decision_guidance": "Decide whether the supplied detector evidence supports bona fide or spoof. Only one detector score is provided; do not invent acoustic evidence. Abstain when insufficient."
 },
 "questions": [
  "decision",
  "evidence_quality",
  "ambiguity"
 ]
}
{
 "type": "choice",
 "choice": "bonafide",
 "confidence": 0.95,
 "probabilities": {
  "bonafide": 0.97,
  "spoof": 0.01,
  "uncertain": 0.02
 }
}


JEV agrees with D2 on every clip: 100% coverage, nothing corrected, nothing introduced. The request explains
why. The context is one score, its threshold, and D2's own verdict (`base_verdict`), so the only consistent
answer is to repeat that verdict.

## 1.5 JEV with richer evidence
Each request carries every detector's calibrated P(bona fide), what we measured about each detector's weak spots,
and the audio facts from 1.2. It never sees the dataset name, clip ID or label. Raw D2 tokens or attention weights
are not sent: they are learned 128-number projections with no meaning a reader (or an LLM) can map to acoustics.

`RUN_JEV = False` keeps this free; set it to True to call the API (one paid call per clip not already cached in
`data/cache/jev/notebook06/`).

In [6]:
RUN_JEV = False
CACHE = Path("../data/cache/jev/notebook06")
GUIDANCE = (
    "You see evidence about one speech clip from four spoofing detectors and simple audio measurements; you do not "
    "hear the audio. Detector reliability measured on unseen web audio and newer spoofing attacks: A2 and D2 are the "
    "most reliable (~10-20% error); D3 is similar on studio audio but less reliable on web audio; A1 is unreliable "
    "on compressed or web audio (~30-58% error). bandwidth_khz near 8 means studio-quality audio; below ~4.5 means "
    "telephone or narrowband audio, where the LFCC-based A1 is least trustworthy. Leading silence is not evidence "
    "either way. Answer 'uncertain' when the reliable detectors disagree or are near 0.5.")
QUESTIONS = {
    "decision": {"type": "choice", "instructions": "Which verdict does the evidence support?",
                 "criteria": {"bonafide": "Reliable evidence favours authentic speech.",
                              "spoof": "Reliable evidence favours spoofed speech.",
                              "uncertain": "Reliable detectors conflict or are inconclusive."}},
    "evidence_quality": {"type": "score", "instructions": "How consistent is the reliable evidence?",
                         "criteria": ["Conflicting", "Mixed", "Consistent"]},
}

FACTS = ["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]

def context(r):  # evidence for one clip; no dataset name, clip id or label
    return {**{f"p_bonafide_{name.split()[0]}": round(float(r[f"p_{m}"]), 3) for m, name in DETECTORS.items()},
            **{k: float(r[k]) for k in FACTS}}

def ask(ctx):
    payload = {"context": ctx, "prompt": GUIDANCE, "questions": QUESTIONS}
    path = CACHE / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest() + ".json")
    if path.exists():
        return json.loads(path.read_text())
    if not RUN_JEV:
        return None
    answers = decide(JEVState(context=ctx, prompt=GUIDANCE, questions=QUESTIONS)).answers
    answers = answers if isinstance(answers, dict) else json.loads(json.dumps(answers, default=lambda o: o.__dict__))
    CACHE.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(answers))
    return answers

replies = [ask(context(r)) for r in pool.to_dict("records")]
print(f"{sum(x is not None for x in replies)}/{len(replies)} JEV answers available")

0/60 JEV answers available


Same clips for every policy, per group: random clips and the hard disagreement clips. `D2 alone` is also
FP1's answer, since 1.4 shows JEV repeats D2.

In [7]:
if all(x is not None for x in replies):
    policies["JEV (rich evidence)"] = np.array([x["decision"]["choice"] for x in replies])
pd.concat({g: pd.DataFrame({k: summarise(v, (pool.group == g).to_numpy()) for k, v in policies.items()}).T
           for g in ["random", "disagree"]}).round(3)

coverage  accuracy on decided  errors  \
random   D2 alone                    1.000                0.800     6.0   
         WavLM average               1.000                0.733     8.0   
         abstain on disagreement     0.933                0.821     5.0   
disagree D2 alone                    1.000                0.433    17.0   
         WavLM average               1.000                0.567    13.0   
         abstain on disagreement     0.000                0.000     0.0   

                                  abstained  
random   D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement        2.0  
disagree D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement       30.0

### Section 1 result
- **FP1 adds nothing.** Given one detector's score and verdict, JEV repeats the verdict on all 180 clips (1.4).
- **Richer evidence:** JEV answers were not cached in this saved run (0/60), so 1.5 has no JEV row yet.
- **The disagreements carry the signal.** On the 1.1 clips, abstaining whenever D2 and A2 disagree lifts accuracy on
  decided clips from 0.62 (D2 alone, i.e. FP1) to 0.82, at 47% coverage. Section 2 tries to resolve those
  disagreements instead of only abstaining.

## Section 2: arbitration between detectors
A1 LFCC-GMM, A2 WavLM and A4 LFCC-MLP judge the same In-the-Wild clip; when they disagree, a judge decides which
verdict to trust. There is one arbitrator (`modules/arbitrator.py`); here we only change what it is asked.

**How to iterate:** copy the last *Try* cell, edit `PROMPT`, `QUESTION` or `context`, and run it. Each try runs
LAYA (local, free) on the same 10 clips, with the detectors listed in two orders. A judge that reads the evidence
gets the clip right and does not flip when only the order changes. Keep every try, so Run All replays them.

Clips: 5 bona fide (including `11249.wav`) and 5 spoof from the fixed arbitration test case
(notebook 05). Thresholds: each artifact's dev
EER; P(bona fide): Platt on dev; `error_rate_pct`: EER on 2021 LA eval.

In [2]:
from audio_df.modules import laya
from audio_df.modules.jev import JEVState
from audio_df.pipelines.arbitration import ORDERS, load_cases

cases, det = load_cases(Path(".."))
try_clips = pd.concat([cases[cases.label == "bonafide"].head(5), cases[cases.label == "spoof"].head(5)])

def try_request(prompt, question, context):
    rows = []
    for r in try_clips.to_dict("records"):
        answers = {o: laya.decide(JEVState(context(r, order), prompt, question))["answers"]["decision"]
                   for o, order in ORDERS.items()}
        first = answers["A1 first"]
        rows.append({"clip": r["clip_id"].removeprefix("ITW_"), "truth": r["label"],
                     **{m.upper(): r[f"v_{m}"] for m in det}, "judge": first["choice"],
                     **{f"P {k}": v for k, v in first["probabilities"].items()},
                     "flips": answers["A1 last"]["choice"] != first["choice"]})
    table = pd.DataFrame(rows).round(2)
    print(f"right {(table.judge == table.truth).sum()}/{len(table)} | flips {table.flips.sum()} | "
          f"uncertain {(table.judge == 'uncertain').sum()}")
    return table

### Try 1: the current reliability request
Each detector's verdict, P(bona fide) and error rate, listed by name.

In [3]:
PROMPT = ("Several spoofing detectors judged the same clip and disagree. Decide which verdict to trust. "
          "Trust detectors with a lower error_rate_pct more.")
QUESTION = {"decision": {"type": "choice", "instructions": "Which verdict should be trusted for this clip?",
                         "criteria": {"bonafide": "The trustworthy evidence says authentic speech.",
                                      "spoof": "The trustworthy evidence says spoofed speech.",
                                      "uncertain": "The evidence conflicts too much to decide."}}}

def context(r, order):
    return {det[m]["name"]: {"verdict": r[f"v_{m}"], "p_bonafide": round(float(r[f"p_{m}"]), 3),
                             "error_rate_pct": det[m]["error_rate_pct"]} for m in order}

try_request(PROMPT, QUESTION, context)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1520.34it/s]

/Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2/SystemCode/.venv/lib/python3.11/site-packages/laya/agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


right 3/10 | flips 5 | uncertain 0


,clip,truth,A1,A2,A4,judge,P bonafide,P spoof,P uncertain,flips
0,11249,bonafide,bonafide,spoof,bonafide,bonafide,0.49,0.30,0.20,True
1,3190,bonafide,spoof,spoof,bonafide,spoof,0.10,0.81,0.09,False
2,19614,bonafide,spoof,bonafide,spoof,spoof,0.14,0.74,0.12,True
3,25652,bonafide,spoof,spoof,bonafide,spoof,0.09,0.83,0.08,False
4,6867,bonafide,spoof,bonafide,bonafide,spoof,0.10,0.82,0.08,False
5,310,spoof,spoof,spoof,bonafide,spoof,0.09,0.83,0.09,False
6,29329,spoof,bonafide,spoof,bonafide,bonafide,0.47,0.32,0.21,True
7,26539,spoof,spoof,spoof,bonafide,spoof,0.09,0.81,0.09,False
8,19501,spoof,bonafide,spoof,bonafide,bonafide,0.49,0.31,0.20,True
9,22426,spoof,bonafide,spoof,bonafide,bonafide,0.50,0.30,0.20,True
